# 🎬 AI Video Dubber v5.0 — Production Pipeline

**Professional AI dubbing** that translates and dubs Chinese/Asian drama videos into 18+ languages.

### ✨ What's New in v5.0

| Feature | Details |
|---------|--------|
| 🎵 **Demucs** | Removes Chinese vocals, keeps clean background music |
| 🐟 **Fish Audio** | Premium TTS with 2M+ voices (optional — falls back to Edge TTS) |
| 🧠 **Smart Director** | Word-level timestamps + audio energy for accurate speaker detection |
| 🎤 **Voice Catalog** | Regional native voices (Hindi voices that sound Indian, not Chinese!) |
| 🎭 **Emotion TTS** | Angry scenes sound angry, sad scenes sound sad |
| 📊 **EBU R128** | Professional loudness normalization |
| 🌐 **18+ Languages** | Hindi, Nepali, Tamil, Telugu, Bengali, English, + more |

### ⚡ Requirements
- **GPU Runtime**: Required for Demucs (T4 free tier works perfectly)
- **Groq API Key**: Free at https://console.groq.com/keys
- **Fish Audio Key** (optional): For premium TTS at https://fish.audio

---

## Step 1: Setup Environment

In [ ]:
# ⚡ IMPORTANT: Enable GPU first!
# Go to: Runtime → Change runtime type → T4 GPU

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || print('⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/DubbedVideos'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'✅ Google Drive mounted! Output: {DRIVE_OUTPUT}')

In [ ]:
# Install all dependencies
!pip install -q groq>=0.11.0 edge-tts>=6.1.0 yt-dlp indic-transliteration>=2.3.0 nest-asyncio>=1.5.0
!pip install -q demucs httpx torchaudio

# Apply nest_asyncio fix for Colab's event loop
import nest_asyncio
nest_asyncio.apply()

# Verify
!ffmpeg -version | head -1
!yt-dlp --version

import torch
print(f'\n✅ PyTorch: {torch.__version__}')
print(f'✅ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f}GB')
print('\n✅ All dependencies installed!')

## Step 2: Clone Dubber Agent

In [ ]:
# Clone the repo (always fresh)
REPO_DIR = '/content/chinese-drama-dubber'
if os.path.exists(REPO_DIR):
    import shutil
    shutil.rmtree(REPO_DIR)
!git clone https://github.com/Dipesh600/chinese-drama-dubber.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'\n📁 Working directory: {os.getcwd()}')

# Show modules
modules = [f for f in os.listdir('.') if f.endswith('.py')]
print(f'\n🧩 Modules ({len(modules)}):')
for m in sorted(modules):
    print(f'   • {m}')
print('\n✅ Repository cloned!')

## Step 3: Set API Keys

| Key | Required | Get it at |
|-----|----------|----------|
| **Groq** | ✅ Yes | https://console.groq.com/keys |
| **Fish Audio** | ❌ Optional | https://fish.audio (premium TTS) |

In [ ]:
# Load API keys
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ Groq API key loaded from Colab Secrets!')
except:
    import getpass
    os.environ['GROQ_API_KEY'] = getpass.getpass('Enter your Groq API Key: ')
    print('✅ Groq API key set!')

# Optional: Fish Audio key for premium TTS
try:
    from google.colab import userdata
    fish_key = userdata.get('FISH_AUDIO_API_KEY')
    if fish_key:
        os.environ['FISH_AUDIO_API_KEY'] = fish_key
        print('✅ Fish Audio API key loaded!')
    else:
        print('ℹ️ No Fish Audio key — using Edge TTS (still good quality)')
except:
    print('ℹ️ No Fish Audio key — using Edge TTS (still good quality)')

# Verify keys
groq_key = os.environ.get('GROQ_API_KEY', '')
fish_key = os.environ.get('FISH_AUDIO_API_KEY', '')
print(f'\n🔑 Groq:       {groq_key[:8]}...{groq_key[-4:]}' if len(groq_key) > 12 else '❌ No Groq key!')
print(f'🐟 Fish Audio: {fish_key[:8]}...{fish_key[-4:]}' if len(fish_key) > 12 else '🔊 Edge TTS mode')

## Step 4: 🎬 Dub Your Video!

### Supported Languages
Hindi, Nepali, Tamil, Telugu, Bengali, Marathi, Gujarati, Kannada, Malayalam, Urdu, English, Spanish, French, Portuguese, German, Japanese, Korean, Arabic, Turkish

In [ ]:
#@title 🎬 Dubbing Configuration { display-mode: "form" }

#@markdown ### Video Settings
VIDEO_URL = 'https://youtu.be/m_mJhZuM4Bk' #@param {type:"string"}
TARGET_LANGUAGE = 'Hindi' #@param ['Hindi', 'Nepali', 'Tamil', 'Telugu', 'Bengali', 'Marathi', 'Gujarati', 'Kannada', 'Malayalam', 'Urdu', 'English', 'Spanish', 'French', 'Portuguese', 'German', 'Japanese', 'Korean', 'Arabic', 'Turkish']
SOURCE_LANGUAGE = 'zh' #@param ['zh', 'en', 'ja', 'ko', 'auto']

#@markdown ### Audio Settings
KEEP_BACKGROUND_MUSIC = True #@param {type:"boolean"}
USE_DEMUCS = True #@param {type:"boolean"}
USE_FISH_AUDIO = True #@param {type:"boolean"}

#@markdown ### Description (helps the AI understand context)
VIDEO_DESCRIPTION = 'Chinese drama video' #@param {type:"string"}

print(f'🎬 Video:      {VIDEO_URL}')
print(f'🌐 Language:   {TARGET_LANGUAGE}')
print(f'🎵 BG Music:   {"Demucs Separation + Smart Ducking" if USE_DEMUCS else "Smart Ducking"}')
print(f'🔊 TTS Engine: {"Fish Audio (premium)" if USE_FISH_AUDIO and os.environ.get("FISH_AUDIO_API_KEY") else "Edge TTS"}')
print(f'📁 Output:     {DRIVE_OUTPUT}')
print(f'\n⏳ Starting v5.0 production pipeline...')
print('=' * 70)

In [ ]:
# Run the v5.0 dubbing pipeline!
import sys, importlib
sys.path.insert(0, REPO_DIR)

# Force reload all modules to pick up v5.0 changes
for mod_name in list(sys.modules.keys()):
    if mod_name in ['orchestrator', 'director', 'preprocessor', 'translator',
                    'dialogue_writer', 'sentence_merger', 'voice_caster',
                    'voice_catalog', 'tts_engine', 'assembler', 'romanizer',
                    'validator', 'audio_separator']:
        del sys.modules[mod_name]

from orchestrator import run_agent

result = run_agent(
    url=VIDEO_URL,
    target_lang=TARGET_LANGUAGE,
    source_lang=SOURCE_LANGUAGE,
    user_description=VIDEO_DESCRIPTION,
    output_dir=DRIVE_OUTPUT,
    preserve_bg=KEEP_BACKGROUND_MUSIC,
    use_fish_audio=USE_FISH_AUDIO,
    use_demucs=USE_DEMUCS
)

if result['success']:
    print('\n' + '=' * 70)
    print(f'  ✅ DUBBING COMPLETE!')
    print(f'  📹 Video:      {result["video_path"]}')
    print(f'  📝 Subtitles:  {result["srt_path"]}')
    print(f'  🎭 Content:    {result.get("content_type", "?")} | {result.get("real_speaker_count", "?")} speakers')
    print(f'  ⏱️ Time:       {result.get("processing_time", "?")}s ({result.get("processing_time", 0)/60:.1f}min)')
    print(f'  💾 Size:       {result.get("size_mb", "?")}MB')
    print(f'  🔊 TTS:        {result.get("tts_engine", "edge_tts")}')
    print(f'  🎵 Demucs:     {"✓ Clean BG" if result.get("demucs_used") else "Not used"}')
    print(f'  🗂️ Saved to:   Google Drive → DubbedVideos/')
    print('=' * 70)
else:
    print(f'\n❌ FAILED at stage {result.get("stage", "?")}: {result.get("error", "Unknown error")}')
    print(f'   Work dir: {result.get("work_dir", "?")}')
    print(f'\n💡 Tip: Check the error above, fix the issue, and re-run this cell.')
    print(f'   The pipeline will resume from where it left off (crash recovery).')

## Step 5: Preview & Download

In [ ]:
# Play the dubbed video right here in Colab!
if result.get('success'):
    from IPython.display import Video, display, HTML
    video_path = result['video_path']

    # Show video player
    try:
        display(Video(video_path, embed=True, width=640))
    except:
        print(f'Video saved at: {video_path}')
        print('Open Google Drive → DubbedVideos to find your video!')

    # Show subtitles preview
    srt_path = result.get('srt_path', '')
    if srt_path and os.path.exists(srt_path):
        print('\n📝 Subtitle Preview (first 15 lines):')
        print('-' * 40)
        with open(srt_path, encoding='utf-8') as f:
            lines = f.readlines()[:15]
            print(''.join(lines))
else:
    print('No video to preview — dubbing did not succeed.')

In [ ]:
# Copy final output to a clean Google Drive folder
if result.get('success'):
    import shutil, time

    ts = time.strftime('%Y%m%d_%H%M%S')
    clean_name = f'dubbed_{TARGET_LANGUAGE}_{ts}'
    final_dir = os.path.join(DRIVE_OUTPUT, clean_name)
    os.makedirs(final_dir, exist_ok=True)

    video_final = os.path.join(final_dir, f'dubbed_{TARGET_LANGUAGE}.mp4')
    srt_final = os.path.join(final_dir, f'subtitles_{TARGET_LANGUAGE}.srt')

    shutil.copy2(result['video_path'], video_final)
    if os.path.exists(result.get('srt_path', '')):
        shutil.copy2(result['srt_path'], srt_final)

    print(f'✅ Clean output saved to Google Drive:')
    print(f'   📹 {video_final}')
    print(f'   📝 {srt_final}')
    print(f'\n📂 Find it in: Google Drive → DubbedVideos → {clean_name}/')
else:
    print('Nothing to copy.')

---
## 🔄 Dub Another Video

Just go back to **Step 4**, change the `VIDEO_URL` and `TARGET_LANGUAGE`, and run again!

---

### 💡 Tips
- **GPU Required**: Demucs needs T4 GPU (Runtime → Change runtime type → T4 GPU)
- **Colab Secrets**: Go to 🔑 icon in left sidebar → Add `GROQ_API_KEY` and optionally `FISH_AUDIO_API_KEY`
- **No GPU?**: Set `USE_DEMUCS = False` to skip audio separation (still works, just no vocal removal)
- **Rate limits**: If Groq rate limits, wait a few minutes and re-run (pipeline resumes from last stage)
- **Fish Audio**: Sign up at https://fish.audio for free credits. Premium TTS with emotion + 2M voices